# Spotify Embedding Regression

This notebook tests how much of the variation in playlist success is explained by:
- structural playlist features
- embedding-based content features
- both together

## Main outcome
`log1p(non_owner_monthly_stream30s)`

## Main analytical population
`User-owned playlists`

The pooled sample is used only as a secondary comparison.

## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


def resolve_path(filename):
    candidates = [
        Path('interview_prep/data_science_assignment') / filename,
        Path(filename),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(filename)

DATA_PATH = resolve_path('spotify_matching_input.csv')

## 2. Load The Embedding-Based Dataset

In [2]:
df = pd.read_csv(DATA_PATH)
user_df = df[df['segment'] == 'User-owned'].copy()
content_pc_cols = [col for col in df.columns if col.startswith('content_pc')]

print('Full sample shape:', df.shape)
print('User-owned sample shape:', user_df.shape)
print('Content PCs:', content_pc_cols)

Full sample shape: (413767, 23)
User-owned sample shape: (413368, 23)
Content PCs: ['content_pc1', 'content_pc2', 'content_pc3', 'content_pc4', 'content_pc5']


## 3. Define Nested Model Formulas

We compare three user-owned models:
- structural only
- embeddings only
- structural + embeddings

Then we repeat a small pooled comparison as a sensitivity check.

In [3]:
structural_formula = (
    'log_non_owner_monthly_stream30s ~ '
    'log_n_tracks + log_n_local_tracks + log_n_artists + log_n_albums'
)

embedding_formula = (
    'log_non_owner_monthly_stream30s ~ ' + ' + '.join(content_pc_cols)
)

combined_formula = (
    structural_formula + ' + ' + ' + '.join(content_pc_cols)
)

pooled_formula = combined_formula + ' + is_spotify_owned'

structural_formula, embedding_formula, combined_formula, pooled_formula

('log_non_owner_monthly_stream30s ~ log_n_tracks + log_n_local_tracks + log_n_artists + log_n_albums',
 'log_non_owner_monthly_stream30s ~ content_pc1 + content_pc2 + content_pc3 + content_pc4 + content_pc5',
 'log_non_owner_monthly_stream30s ~ log_n_tracks + log_n_local_tracks + log_n_artists + log_n_albums + content_pc1 + content_pc2 + content_pc3 + content_pc4 + content_pc5',
 'log_non_owner_monthly_stream30s ~ log_n_tracks + log_n_local_tracks + log_n_artists + log_n_albums + content_pc1 + content_pc2 + content_pc3 + content_pc4 + content_pc5 + is_spotify_owned')

## 4. Fit User-Owned OLS Models

In [4]:
user_structural = smf.ols(structural_formula, data=user_df).fit()
user_embedding = smf.ols(embedding_formula, data=user_df).fit()
user_combined = smf.ols(combined_formula, data=user_df).fit()

user_model_summary = pd.DataFrame([
    {
        'model': 'Structural only',
        'r2': user_structural.rsquared,
        'adj_r2': user_structural.rsquared_adj,
        'aic': user_structural.aic,
    },
    {
        'model': 'Embeddings only',
        'r2': user_embedding.rsquared,
        'adj_r2': user_embedding.rsquared_adj,
        'aic': user_embedding.aic,
    },
    {
        'model': 'Structural + embeddings',
        'r2': user_combined.rsquared,
        'adj_r2': user_combined.rsquared_adj,
        'aic': user_combined.aic,
    },
]).sort_values('adj_r2', ascending=False)

user_model_summary

,model,r2,adj_r2,aic
2,Structural + embeddings,0.099900,0.099880,1.535885e+06
0,Structural only,0.084906,0.084897,1.542705e+06
1,Embeddings only,0.037552,0.037541,1.563562e+06


## 5. Test The Incremental Value Of Embeddings

In [5]:
user_nested_test = anova_lm(user_structural, user_combined)
user_nested_test

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,413363.0,1.010776e+06,0.0,NaN,NaN,NaN
1,413358.0,9.942147e+05,5.0,16561.331224,1377.118846,0.0


## 6. Out-of-Sample Test R²

A simple 80/20 holdout check shows whether the content PCs improve predictive performance, not just in-sample fit.

In [6]:
feature_sets = {
    'Structural only': ['log_n_tracks', 'log_n_local_tracks', 'log_n_artists', 'log_n_albums'],
    'Embeddings only': content_pc_cols,
    'Structural + embeddings': ['log_n_tracks', 'log_n_local_tracks', 'log_n_artists', 'log_n_albums'] + content_pc_cols,
}

train_df, test_df = train_test_split(user_df, test_size=0.2, random_state=42)

holdout_rows = []
for model_name, cols in feature_sets.items():
    lr = LinearRegression()
    lr.fit(train_df[cols], train_df['log_non_owner_monthly_stream30s'])
    test_pred = lr.predict(test_df[cols])
    holdout_rows.append({
        'model': model_name,
        'test_r2': r2_score(test_df['log_non_owner_monthly_stream30s'], test_pred),
    })

holdout_summary = pd.DataFrame(holdout_rows).sort_values('test_r2', ascending=False)
holdout_summary

,model,test_r2
2,Structural + embeddings,0.100326
0,Structural only,0.086755
1,Embeddings only,0.036128


## 7. Key Coefficients From The Combined User-Owned Model

In [7]:
coef_table = pd.DataFrame({
    'term': user_combined.params.index,
    'coef': user_combined.params.values,
    'pvalue': user_combined.pvalues.values,
})

coef_table['abs_coef'] = coef_table['coef'].abs()
coef_table.sort_values('abs_coef', ascending=False).head(15)

,term,coef,pvalue,abs_coef
7,content_pc3,-1.795913,0.000000e+00,1.795913
0,Intercept,1.640378,0.000000e+00,1.640378
9,content_pc5,-0.849387,1.618540e-205,0.849387
6,content_pc2,0.723125,4.701725e-261,0.723125
1,log_n_tracks,0.396070,0.000000e+00,0.396070
2,log_n_local_tracks,-0.079182,9.275674e-152,0.079182
8,content_pc4,-0.068345,1.105194e-02,0.068345
5,content_pc1,-0.052356,2.351367e-03,0.052356
4,log_n_albums,-0.040637,7.185303e-19,0.040637
3,log_n_artists,-0.014687,2.270044e-04,0.014687


## 8. Pooled Sensitivity Check

In [8]:
pooled_structural = smf.ols(structural_formula, data=df).fit()
pooled_combined = smf.ols(combined_formula, data=df).fit()
pooled_plus_spotify = smf.ols(pooled_formula, data=df).fit()

pooled_summary = pd.DataFrame([
    {
        'model': 'Pooled structural only',
        'adj_r2': pooled_structural.rsquared_adj,
    },
    {
        'model': 'Pooled structural + embeddings',
        'adj_r2': pooled_combined.rsquared_adj,
    },
    {
        'model': 'Pooled structural + embeddings + Spotify indicator',
        'adj_r2': pooled_plus_spotify.rsquared_adj,
    },
])

pooled_summary

,model,adj_r2
0,Pooled structural only,0.082296
1,Pooled structural + embeddings,0.097253
2,Pooled structural + embeddings + Spotify indic...,0.123233


## 9. Readout

A good way to interpret these models:
- structural features explain a meaningful but limited share of variation
- embedding-based content features add incremental signal
- the combined model is better than structure alone, but most variation remains unexplained
- in the pooled sample, a Spotify-owned indicator still adds a large jump in explanatory power